In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
from typing import Dict, List, Any

## 1. Load Sample Data

In [ ]:
# Load the demo sales dataset
df = pd.read_csv("../data/demo_sales.csv")
print(f"✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Data Profiler Class

In [ ]:
class DataProfiler:
    """
    A simple data profiler for exploratory data analysis.
    Generates comprehensive statistics about the dataset.
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def basic_info(self) -> Dict[str, Any]:
        """Get basic dataset information."""
        return {
            "rows": len(self.df),
            "columns": len(self.df.columns),
            "memory_mb": self.df.memory_usage(deep=True).sum() / 1024**2,
            "duplicates": self.df.duplicated().sum(),
            "total_missing": self.df.isnull().sum().sum(),
            "missing_pct": (self.df.isnull().sum().sum() / self.df.size) * 100
        }
    
    def column_types(self) -> pd.DataFrame:
        """Analyze column types."""
        type_info = []
        for col in self.df.columns:
            type_info.append({
                "column": col,
                "dtype": str(self.df[col].dtype),
                "non_null": self.df[col].notna().sum(),
                "null": self.df[col].isnull().sum(),
                "unique": self.df[col].nunique(),
                "sample": str(self.df[col].dropna().iloc[0]) if len(self.df[col].dropna()) > 0 else None
            })
        return pd.DataFrame(type_info)
    
    def missing_analysis(self) -> pd.DataFrame:
        """Analyze missing values."""
        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df)) * 100
        
        result = pd.DataFrame({
            "column": missing.index,
            "missing_count": missing.values,
            "missing_pct": missing_pct.values
        })
        return result[result["missing_count"] > 0].sort_values("missing_pct", ascending=False)
    
    def numeric_summary(self) -> pd.DataFrame:
        """Statistical summary for numeric columns."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) == 0:
            return pd.DataFrame()
        
        summary = self.df[numeric_cols].describe().T
        summary["skew"] = self.df[numeric_cols].skew()
        summary["kurtosis"] = self.df[numeric_cols].kurtosis()
        return summary
    
    def categorical_summary(self) -> pd.DataFrame:
        """Summary for categorical columns."""
        cat_cols = self.df.select_dtypes(include=["object", "category"]).columns
        if len(cat_cols) == 0:
            return pd.DataFrame()
        
        cat_info = []
        for col in cat_cols:
            value_counts = self.df[col].value_counts()
            cat_info.append({
                "column": col,
                "unique": self.df[col].nunique(),
                "top_value": value_counts.index[0] if len(value_counts) > 0 else None,
                "top_freq": value_counts.iloc[0] if len(value_counts) > 0 else 0,
                "top_pct": (value_counts.iloc[0] / len(self.df)) * 100 if len(value_counts) > 0 else 0
            })
        return pd.DataFrame(cat_info)
    
    def generate_report(self) -> str:
        """Generate a text report."""
        info = self.basic_info()
        
        report = []
        report.append("=" * 50)
        report.append("📊 DATA PROFILE REPORT")
        report.append("=" * 50)
        report.append(f"\n📋 Basic Information:")
        report.append(f"  - Rows: {info['rows']:,}")
        report.append(f"  - Columns: {info['columns']}")
        report.append(f"  - Memory: {info['memory_mb']:.2f} MB")
        report.append(f"  - Duplicates: {info['duplicates']}")
        report.append(f"  - Missing: {info['missing_pct']:.1f}%")
        
        # Column types
        report.append(f"\n📊 Column Types:")
        for dtype, count in self.df.dtypes.value_counts().items():
            report.append(f"  - {dtype}: {count} columns")
        
        return "\n".join(report)

## 3. Run the Profiler

In [ ]:
# Create profiler
profiler = DataProfiler(df)

# Print report
print(profiler.generate_report())

In [ ]:
# Column types analysis
print("\n📋 Column Details:")
profiler.column_types()

In [ ]:
# Numeric summary
print("\n📈 Numeric Columns Summary:")
profiler.numeric_summary()

In [ ]:
# Categorical summary
print("\n📝 Categorical Columns Summary:")
profiler.categorical_summary()

## 4. Data Quality Checker

In [ ]:
class DataQualityChecker:
    """Check data quality and identify potential issues."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.warnings = []
    
    def check_all(self) -> List[str]:
        """Run all quality checks."""
        self.warnings = []
        self._check_missing()
        self._check_duplicates()
        self._check_constants()
        self._check_high_cardinality()
        self._check_outliers()
        return self.warnings
    
    def _check_missing(self, threshold: float = 0.3):
        """Check for columns with high missing values."""
        missing_pct = self.df.isnull().sum() / len(self.df)
        high_missing = missing_pct[missing_pct > threshold]
        for col, pct in high_missing.items():
            self.warnings.append(f"⚠️ HIGH MISSING: '{col}' has {pct:.1%} missing values")
    
    def _check_duplicates(self):
        """Check for duplicate rows."""
        dup_count = self.df.duplicated().sum()
        if dup_count > 0:
            self.warnings.append(f"⚠️ DUPLICATES: {dup_count} duplicate rows found")
    
    def _check_constants(self):
        """Check for constant columns."""
        for col in self.df.columns:
            if self.df[col].nunique() == 1:
                self.warnings.append(f"⚠️ CONSTANT: '{col}' has only 1 unique value")
    
    def _check_high_cardinality(self, threshold: float = 0.9):
        """Check for high cardinality categorical columns."""
        cat_cols = self.df.select_dtypes(include=["object"]).columns
        for col in cat_cols:
            if self.df[col].nunique() / len(self.df) > threshold:
                self.warnings.append(f"⚠️ HIGH CARDINALITY: '{col}' might be an ID column")
    
    def _check_outliers(self, threshold: float = 3.0):
        """Check for outliers using z-score."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            mean = self.df[col].mean()
            std = self.df[col].std()
            if std > 0:
                z_scores = np.abs((self.df[col] - mean) / std)
                outliers = (z_scores > threshold).sum()
                if outliers > 0:
                    self.warnings.append(f"⚠️ OUTLIERS: '{col}' has {outliers} potential outliers")

In [ ]:
# Run quality checks
checker = DataQualityChecker(df)
warnings = checker.check_all()

print("\n🔍 Data Quality Check Results:")
print("=" * 50)
if warnings:
    for w in warnings:
        print(w)
else:
    print("✅ No issues found!")

## 5. Test with Titanic Dataset

In [ ]:
# Load Titanic from URL
titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
print(f"✅ Loaded Titanic: {titanic.shape}")

# Profile it
profiler2 = DataProfiler(titanic)
print(profiler2.generate_report())

# Check quality
checker2 = DataQualityChecker(titanic)
warnings2 = checker2.check_all()
print("\n🔍 Quality Issues:")
for w in warnings2:
    print(w)

## ✅ Summary

This module provides:
- `DataProfiler` - Comprehensive data profiling
  - `basic_info()` - Row/column counts, memory, duplicates
  - `column_types()` - Detailed column analysis
  - `numeric_summary()` - Statistics for numeric columns
  - `categorical_summary()` - Analysis of categorical columns
- `DataQualityChecker` - Identify data quality issues
  - Missing values, duplicates, constants, outliers